# Heart Disease UCI - MLOps Assignment
## End-to-End ML Pipeline: EDA, Feature Engineering, Model Training & MLflow Tracking
**Course:** MLOps (S2-25_AMLCSZG523) | **Dataset:** Heart Disease UCI

## 0. Install Dependencies

In [ ]:
# Run once to install dependencies
# !pip install -r ../requirements.txt

## 1. Imports & Configuration

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.sklearn
import pickle
import json

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    confusion_matrix, roc_curve, ConfusionMatrixDisplay
)
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

RANDOM_STATE = 42
TEST_SIZE = 0.2
DATA_PATH = '../data/heart.csv'
MODEL_DIR = '../src/models/'
os.makedirs(MODEL_DIR, exist_ok=True)

print('All imports successful!')

## 2. Data Acquisition

In [ ]:
import urllib.request

DATA_URL = 'https://raw.githubusercontent.com/dsrscientist/dataset1/master/heart_disease.csv'
os.makedirs('../data', exist_ok=True)

# Download if not present
if not os.path.exists(DATA_PATH):
    print('Downloading dataset...')
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)
    print(f'Dataset saved to {DATA_PATH}')
else:
    print(f'Dataset already exists at {DATA_PATH}')

df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# --- Basic Info ---
print('=== Dataset Info ===')
print(df.info())
print('\n=== Statistical Summary ===')
df.describe().T

In [ ]:
# --- Missing Values ---
missing = df.isnull().sum()
print('=== Missing Values ===')
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values found!')

In [ ]:
# Standardize target column name
if 'target' not in df.columns:
    df = df.rename(columns={df.columns[-1]: 'target'})

# Binarize target if needed (some versions have 0-4)
df['target'] = (df['target'] > 0).astype(int)

# --- Class Balance ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['target'].value_counts()
axes[0].bar(['No Disease (0)', 'Disease (1)'], counts.values, color=['#4C72B0', '#DD8452'], edgecolor='black')
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 1, str(v), ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=['No Disease', 'Disease'], autopct='%1.1f%%',
            colors=['#4C72B0', '#DD8452'], startangle=90, explode=(0.05, 0.05))
axes[1].set_title('Class Balance (Pie)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../screenshots/class_balance.png', bbox_inches='tight')
plt.show()
print(f'Class balance ratio: {counts[0]/counts[1]:.2f}')

In [ ]:
# --- Feature Histograms ---
os.makedirs('../screenshots', exist_ok=True)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

fig, axes = plt.subplots(3, 5, figsize=(20, 12))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    if i < len(axes):
        axes[i].hist(df[col], bins=20, color='#4C72B0', edgecolor='white', alpha=0.85)
        axes[i].set_title(col, fontsize=11, fontweight='bold')
        axes[i].set_xlabel('Value')
        axes[i].set_ylabel('Frequency')

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Distributions', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../screenshots/feature_histograms.png', bbox_inches='tight')
plt.show()

In [ ]:
# --- Correlation Heatmap ---
fig, ax = plt.subplots(figsize=(14, 10))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Heatmap', fontsize=15, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('../screenshots/correlation_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# --- Feature vs Target Boxplots ---
continuous_features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
available = [c for c in continuous_features if c in df.columns]

fig, axes = plt.subplots(1, len(available), figsize=(16, 5))
for i, col in enumerate(available):
    df.boxplot(column=col, by='target', ax=axes[i],
               boxprops=dict(color='#4C72B0'),
               medianprops=dict(color='red', linewidth=2))
    axes[i].set_title(col, fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Target (0=No Disease, 1=Disease)')

plt.suptitle('Feature Distribution by Target Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../screenshots/boxplots_by_target.png', bbox_inches='tight')
plt.show()

## 4. Feature Engineering & Preprocessing

In [ ]:
# --- Define Feature Types ---
CATEGORICAL_FEATURES = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
NUMERICAL_FEATURES   = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
TARGET = 'target'

# Keep only features that exist in the dataframe
cat_feats = [f for f in CATEGORICAL_FEATURES if f in df.columns]
num_feats = [f for f in NUMERICAL_FEATURES   if f in df.columns]
all_feats = num_feats + cat_feats

print(f'Numerical features: {num_feats}')
print(f'Categorical features: {cat_feats}')

X = df[all_feats]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print(f'\nTrain size: {X_train.shape}, Test size: {X_test.shape}')

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# --- Full Preprocessing Pipeline ---
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, num_feats),
    ('cat', categorical_transformer, cat_feats)
])

print('Preprocessing pipeline defined!')
print(preprocessor)

## 5. Model Training with MLflow Experiment Tracking

In [ ]:
# --- MLflow Setup ---
mlflow.set_tracking_uri('file:../mlruns')
mlflow.set_experiment('Heart-Disease-Classification')

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def evaluate_model(model_pipeline, X_tr, y_tr, X_te, y_te):
    """Fit, predict and return metrics dict."""
    model_pipeline.fit(X_tr, y_tr)
    y_pred  = model_pipeline.predict(X_te)
    y_proba = model_pipeline.predict_proba(X_te)[:, 1]
    
    cv_scores = cross_val_score(model_pipeline, X_tr, y_tr, cv=cv, scoring='roc_auc')
    
    return {
        'accuracy':  accuracy_score(y_te, y_pred),
        'precision': precision_score(y_te, y_pred),
        'recall':    recall_score(y_te, y_pred),
        'f1':        f1_score(y_te, y_pred),
        'roc_auc':   roc_auc_score(y_te, y_proba),
        'cv_roc_auc_mean': cv_scores.mean(),
        'cv_roc_auc_std':  cv_scores.std(),
        'y_pred':  y_pred,
        'y_proba': y_proba
    }

print('MLflow tracking URI:', mlflow.get_tracking_uri())

In [ ]:
# --- Model 1: Logistic Regression ---
lr_params = {'classifier__C': 1.0, 'classifier__max_iter': 1000, 'classifier__solver': 'lbfgs'}

lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier',   LogisticRegression(random_state=RANDOM_STATE, **{k.replace('classifier__',''):v for k,v in lr_params.items()}))
])

with mlflow.start_run(run_name='LogisticRegression'):
    mlflow.log_params(lr_params)
    mlflow.log_param('model_type', 'LogisticRegression')
    mlflow.log_param('test_size',  TEST_SIZE)
    mlflow.log_param('random_state', RANDOM_STATE)

    lr_metrics = evaluate_model(lr_pipeline, X_train, y_train, X_test, y_test)

    for k, v in lr_metrics.items():
        if isinstance(v, float):
            mlflow.log_metric(k, v)

    # Confusion matrix plot
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay.from_predictions(y_test, lr_metrics['y_pred'], ax=ax,
                                            display_labels=['No Disease', 'Disease'],
                                            colorbar=False, cmap='Blues')
    ax.set_title('Logistic Regression - Confusion Matrix', fontweight='bold')
    plt.tight_layout()
    plt.savefig('../screenshots/lr_confusion_matrix.png')
    mlflow.log_artifact('../screenshots/lr_confusion_matrix.png')
    plt.show()

    mlflow.sklearn.log_model(lr_pipeline, 'logistic_regression_model')

print(f"LR  => Accuracy: {lr_metrics['accuracy']:.4f} | ROC-AUC: {lr_metrics['roc_auc']:.4f} | CV-AUC: {lr_metrics['cv_roc_auc_mean']:.4f}±{lr_metrics['cv_roc_auc_std']:.4f}")

In [ ]:
# --- Model 2: Random Forest ---
rf_params = {'classifier__n_estimators': 200, 'classifier__max_depth': 8,
             'classifier__min_samples_split': 5, 'classifier__min_samples_leaf': 2}

rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier',   RandomForestClassifier(random_state=RANDOM_STATE, **{k.replace('classifier__',''):v for k,v in rf_params.items()}))
])

with mlflow.start_run(run_name='RandomForest'):
    mlflow.log_params(rf_params)
    mlflow.log_param('model_type', 'RandomForest')
    mlflow.log_param('test_size', TEST_SIZE)

    rf_metrics = evaluate_model(rf_pipeline, X_train, y_train, X_test, y_test)

    for k, v in rf_metrics.items():
        if isinstance(v, float):
            mlflow.log_metric(k, v)

    # Feature importances
    rf_clf = rf_pipeline.named_steps['classifier']
    ohe_cats = rf_pipeline.named_steps['preprocessor']\
               .named_transformers_['cat']['encoder']\
               .get_feature_names_out(cat_feats).tolist()
    feat_names = num_feats + ohe_cats
    importances = pd.Series(rf_clf.feature_importances_, index=feat_names).sort_values(ascending=False)[:15]

    fig, ax = plt.subplots(figsize=(10, 6))
    importances.plot(kind='barh', ax=ax, color='#4C72B0', edgecolor='white')
    ax.set_title('Random Forest - Top 15 Feature Importances', fontweight='bold', fontsize=13)
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig('../screenshots/rf_feature_importance.png')
    mlflow.log_artifact('../screenshots/rf_feature_importance.png')
    plt.show()

    # Confusion matrix
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay.from_predictions(y_test, rf_metrics['y_pred'], ax=ax,
                                            display_labels=['No Disease', 'Disease'],
                                            colorbar=False, cmap='Greens')
    ax.set_title('Random Forest - Confusion Matrix', fontweight='bold')
    plt.tight_layout()
    plt.savefig('../screenshots/rf_confusion_matrix.png')
    mlflow.log_artifact('../screenshots/rf_confusion_matrix.png')
    plt.show()

    mlflow.sklearn.log_model(rf_pipeline, 'random_forest_model')

print(f"RF  => Accuracy: {rf_metrics['accuracy']:.4f} | ROC-AUC: {rf_metrics['roc_auc']:.4f} | CV-AUC: {rf_metrics['cv_roc_auc_mean']:.4f}±{rf_metrics['cv_roc_auc_std']:.4f}")

In [ ]:
# --- Model 3: Gradient Boosting (Bonus) ---
gb_params = {'classifier__n_estimators': 150, 'classifier__learning_rate': 0.1,
             'classifier__max_depth': 4, 'classifier__subsample': 0.8}

gb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier',   GradientBoostingClassifier(random_state=RANDOM_STATE, **{k.replace('classifier__',''):v for k,v in gb_params.items()}))
])

with mlflow.start_run(run_name='GradientBoosting'):
    mlflow.log_params(gb_params)
    mlflow.log_param('model_type', 'GradientBoosting')

    gb_metrics = evaluate_model(gb_pipeline, X_train, y_train, X_test, y_test)

    for k, v in gb_metrics.items():
        if isinstance(v, float):
            mlflow.log_metric(k, v)

    mlflow.sklearn.log_model(gb_pipeline, 'gradient_boosting_model')

print(f"GB  => Accuracy: {gb_metrics['accuracy']:.4f} | ROC-AUC: {gb_metrics['roc_auc']:.4f} | CV-AUC: {gb_metrics['cv_roc_auc_mean']:.4f}±{gb_metrics['cv_roc_auc_std']:.4f}")

## 6. Model Comparison & ROC Curves

In [ ]:
# --- Comparison Table ---
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'Gradient Boosting'],
    'Accuracy':  [lr_metrics['accuracy'],  rf_metrics['accuracy'],  gb_metrics['accuracy']],
    'Precision': [lr_metrics['precision'], rf_metrics['precision'], gb_metrics['precision']],
    'Recall':    [lr_metrics['recall'],    rf_metrics['recall'],    gb_metrics['recall']],
    'F1':        [lr_metrics['f1'],        rf_metrics['f1'],        gb_metrics['f1']],
    'ROC-AUC':   [lr_metrics['roc_auc'],   rf_metrics['roc_auc'],   gb_metrics['roc_auc']],
    'CV-AUC':    [lr_metrics['cv_roc_auc_mean'], rf_metrics['cv_roc_auc_mean'], gb_metrics['cv_roc_auc_mean']],
}).set_index('Model').round(4)

print('=== Model Comparison ===')
results.style.highlight_max(color='lightgreen').highlight_min(color='lightyellow')

In [ ]:
# --- ROC Curves ---
fig, ax = plt.subplots(figsize=(9, 7))
colors  = ['#4C72B0', '#55A868', '#C44E52']
models  = [('Logistic Regression', lr_metrics), ('Random Forest', rf_metrics), ('Gradient Boosting', gb_metrics)]

for (name, met), color in zip(models, colors):
    fpr, tpr, _ = roc_curve(y_test, met['y_proba'])
    ax.plot(fpr, tpr, color=color, lw=2.5, label=f'{name} (AUC={met["roc_auc"]:.3f})')

ax.plot([0,1],[0,1], 'k--', lw=1.5, label='Random Classifier')
ax.set_xlabel('False Positive Rate', fontsize=13)
ax.set_ylabel('True Positive Rate', fontsize=13)
ax.set_title('ROC Curves - Model Comparison', fontsize=15, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../screenshots/roc_curves_comparison.png', bbox_inches='tight')
plt.show()

## 7. Save Best Model & Pipeline

In [ ]:
# Select best model by ROC-AUC
model_scores = {
    'logistic_regression': (lr_pipeline, lr_metrics['roc_auc']),
    'random_forest':       (rf_pipeline, rf_metrics['roc_auc']),
    'gradient_boosting':   (gb_pipeline, gb_metrics['roc_auc'])
}

best_name, (best_model, best_auc) = max(model_scores.items(), key=lambda x: x[1][1])
print(f'Best model: {best_name} with ROC-AUC = {best_auc:.4f}')

# Save as pickle
model_path = os.path.join(MODEL_DIR, 'best_model.pkl')
with open(model_path, 'wb') as f:
    pickle.dump(best_model, f)
print(f'Model saved to {model_path}')

# Save feature config for API
feature_config = {
    'numerical_features': num_feats,
    'categorical_features': cat_feats,
    'all_features': all_feats,
    'target': TARGET,
    'best_model': best_name,
    'best_roc_auc': round(best_auc, 4)
}
with open(os.path.join(MODEL_DIR, 'feature_config.json'), 'w') as f:
    json.dump(feature_config, f, indent=2)

print('Feature config saved!')
print(json.dumps(feature_config, indent=2))

In [ ]:
# --- Quick Inference Test ---
sample = X_test.iloc[:3]
preds  = best_model.predict(sample)
probas = best_model.predict_proba(sample)[:, 1]

print('=== Sample Inference Test ===')
for i, (pred, prob) in enumerate(zip(preds, probas)):
    label = 'Heart Disease' if pred == 1 else 'No Disease'
    print(f'Sample {i+1}: Prediction={label} | Confidence={prob:.4f}')

print('\nNotebook complete!')